In [58]:
#preapring datasets

import pandas as pd

df=pd.read_csv("train.tsv",sep='\t',header=None)
# print(df.head())

# df.head()
columns = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "job_title",
    "state",
    "party",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts",
    "context"
]
df.columns=columns
df.columns
# print(df.isna().sum())
print(df.shape)
df=df[['statement','label']]
df.dropna(inplace=True)
# print(df.isna().sum())
def convert_label(x):
    if x in ["true", "mostly-true"]:
        return 1
    else:
        return 0

df["label"] = df["label"].apply(convert_label)
print(df.shape)
df['label'].value_counts()
df['statement'].head(6)


(10240, 14)
(10240, 2)


0    Says the Annies List political group supports ...
1    When did the decline of coal start? It started...
2    Hillary Clinton agrees with John McCain "by vo...
3    Health care reform legislation is likely to ma...
4    The economic turnaround started at the end of ...
5    The Chicago Bears have had more starting quart...
Name: statement, dtype: object

In [65]:
#NLP preprocessing

import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text) #removing number with ''
    text = text.translate(str.maketrans("", "", string.punctuation)) #remove puncatations
    text = re.sub(r"\s+", " ", text).strip() #remove the extra spaces
    return text

df["statement"] = df["statement"].apply(clean_text)


##splilting the dataset inorder to tokenized 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["statement"], df["label"], test_size=0.2, random_state=42
)

#tokenIZXATIONS

from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 5000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
print('Tokenize exmaple::')
print(X_train_seq[8])
print(X_train.iloc[8])

Tokenize exmaple::
[7, 93, 134, 118, 258, 1110, 709, 1996, 8, 944, 132, 6, 1050, 74, 4263]
says wisconsin gov scott walker eliminated cancer screenings for uninsured women and offered no alternatives


Text
 ↓
Tokenization → [2, 1, 3]
 ↓
Embedding → [[vector], [vector], [vector]]
 ↓
LSTM → learns patterns

In [66]:
embedding_index = {}

with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = list(map(float, values[1:]))
        embedding_index[word] = vector

import numpy as np

embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        vector = embedding_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector
print(embedding_matrix[1])

[-0.038194 -0.24487   0.72812  -0.39961   0.083172  0.043953 -0.39141
  0.3344   -0.57545   0.087459  0.28787  -0.06731   0.30906  -0.26384
 -0.13231  -0.20757   0.33395  -0.33848  -0.31743  -0.48336   0.1464
 -0.37304   0.34577   0.052041  0.44946  -0.46971   0.02628  -0.54155
 -0.15518  -0.14107  -0.039722  0.28277   0.14393   0.23464  -0.31021
  0.086173  0.20397   0.52624   0.17164  -0.082378 -0.71787  -0.41531
  0.20335  -0.12763   0.41367   0.55187   0.57908  -0.33477  -0.36559
 -0.54857  -0.062892  0.26584   0.30205   0.99775  -0.80481  -3.0243
  0.01254  -0.36942   2.2167    0.72201  -0.24978   0.92136   0.034514
  0.46745   1.1079   -0.19358  -0.074575  0.23353  -0.052062 -0.22044
  0.057162 -0.15806  -0.30798  -0.41625   0.37972   0.15006  -0.53212
 -0.2055   -1.2526    0.071624  0.70565   0.49744  -0.42063   0.26148
 -1.538    -0.30223  -0.073438 -0.28312   0.37104  -0.25217   0.016215
 -0.017099 -0.38984   0.87424  -0.72569  -0.51058  -0.52028  -0.1459
  0.8278    0.27062 ]

In [69]:
# length=[len(seq) for seq in X_test]
print(len)


[10, 6, 26, 24, 9, 19, 27, 22, 15, 17, 20, 12, 32, 16, 13, 9, 22, 9, 19, 24, 21, 20, 26, 26, 14, 17, 8, 32, 9, 14, 12, 20, 18, 13, 7, 16, 10, 12, 31, 13, 20, 11, 11, 15, 17, 18, 14, 12, 11, 24, 6, 15, 38, 16, 14, 16, 9, 18, 17, 28, 45, 17, 9, 19, 13, 16, 10, 24, 16, 18, 18, 43, 12, 25, 20, 13, 10, 12, 9, 18, 14, 10, 17, 11, 10, 11, 10, 16, 18, 19, 26, 19, 15, 6, 13, 6, 19, 22, 28, 13, 10, 13, 26, 22, 15, 13, 26, 11, 14, 10, 7, 19, 16, 25, 21, 20, 31, 13, 19, 20, 9, 13, 19, 17, 24, 20, 14, 19, 16, 18, 17, 19, 23, 20, 15, 9, 10, 25, 13, 21, 5, 22, 7, 13, 9, 16, 17, 15, 30, 23, 15, 17, 13, 17, 15, 18, 13, 12, 14, 11, 25, 17, 21, 16, 20, 29, 16, 21, 36, 16, 19, 12, 34, 39, 7, 25, 18, 13, 12, 13, 16, 5, 15, 10, 25, 12, 17, 25, 12, 19, 7, 10, 15, 8, 21, 27, 24, 12, 18, 10, 14, 8, 14, 24, 14, 13, 17, 14, 22, 7, 8, 18, 10, 9, 9, 13, 19, 23, 18, 10, 5, 18, 11, 22, 6, 14, 15, 9, 23, 15, 6, 8, 11, 14, 7, 6, 7, 23, 20, 12, 19, 9, 10, 21, 24, 20, 17, 14, 7, 13, 25, 28, 16, 39, 16, 22, 32, 19, 15, 2